# Step 03: graph → topics, one round at a time

Each round takes the graph's nodes nearest the seeds first (then in the order of `[topics] reasons`), skips nodes that were a topic in an earlier round, and turns each into a search per source:

- **PubMed**: `"Glioma"[MeSH]`, or `[MeSH:noexp]` when its children are topics too; `[nm]` for supplementary records; for a gene, MeSH's *SYMBOL protein, human* record plus its full name
- **ClinicalTrials.gov**: a condition search for a disease heading, an intervention search for a drug heading

Every search is counted against the live API (it needs network access and an NCBI key, `.env.local` or `~/.ncbi.env`); `take` is what the `[download]` limits will download.

All the example notebooks use `experiments/examples.toml`: the default experiment with tiny limits and its own pool (`corpora/examples`). Run `00-setup` for it first: `python src/00-setup/run.py --experiment experiments/examples.toml --no-index`.

In [1]:
import contextlib
import importlib.util
import io
import sys
import tomllib
from pathlib import Path

import duckdb
import pyarrow as pa

SRC = Path.cwd().parent
LACUNA = SRC.parent
ROOT = LACUNA / 'data'
experiment = tomllib.loads((LACUNA / 'experiments' / 'examples.toml').read_text())


def load(step):
    """A step's run.py as a module (every step's entry point is called run.py)."""
    sys.path.insert(0, str(SRC / step))
    spec = importlib.util.spec_from_file_location(f'step_{step[:2]}', SRC / step / 'run.py')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


query, expand, topics_step = load('01-query'), load('02-expand'), load('03-topics')


def round_of(goal, previous=(), max_topics=6, keep=lambda t: True):
    experiment['topics']['max_topics'] = max_topics
    seeds = query.seeds(goal, experiment, ROOT)
    nodes, edges = expand.build([s['id'] for s in seeds], experiment, ROOT)
    with contextlib.redirect_stdout(io.StringIO()):
        topics, skipped = topics_step.choose(experiment, nodes.to_pylist(), list(previous), ROOT)
    rows = pa.Table.from_pylist([{'name': t['name'], 'reason': t['reason'], 'distance': t['distance'],
                                  'pubmed query': t.get('pubmed', {}).get('query'),
                                  'papers': t.get('pubmed', {}).get('count'), 'take': t['take'],
                                  'trial search': str(t.get('clinicaltrials', {}).get('params')),
                                  'trials': t.get('clinicaltrials', {}).get('count')}
                                 for t in topics if keep(t)])
    print(duckdb.sql('SELECT * FROM rows'))
    for s in skipped:
        print(f"  skipped {s['label']}: {s['why']}")
    return topics

## 1. Rounds: each one carries on where the last stopped

Round 2 is given round 1 as `previous`, so it skips those nodes and takes the next ones. Glioma is searched `[MeSH:noexp]` because its subtypes are topics of their own.

In [2]:
round_1 = round_of('new treatment avenues for glioblastoma')

┌──────────────────────────────────────────────┬───────────────┬──────────┬────────────────────────────────────────────────────┬────────┬───────┬────────────────────────────────────────────────────┬────────┐
│                     name                     │    reason     │ distance │                    pubmed query                    │ papers │ take  │                    trial search                    │ trials │
│                   varchar                    │    varchar    │  int64   │                      varchar                       │ int64  │ int64 │                      varchar                       │ int64  │
├──────────────────────────────────────────────┼───────────────┼──────────┼────────────────────────────────────────────────────┼────────┼───────┼────────────────────────────────────────────────────┼────────┤
│ glioblastoma                                 │ seed          │        0 │ "Glioblastoma"[MeSH]                               │  35028 │     3 │ {'query.cond': 'Gliobl

In [3]:
round_2 = round_of('new treatment avenues for glioblastoma', previous=[{'topics': round_1}])

┌────────────────────┬──────────┬──────────┬───────────────────────────────┬────────┬───────┬──────────────────────────────────────┬────────┐
│        name        │  reason  │ distance │         pubmed query          │ papers │ take  │             trial search             │ trials │
│      varchar       │ varchar  │  int64   │            varchar            │ int64  │ int64 │               varchar                │ int64  │
├────────────────────┼──────────┼──────────┼───────────────────────────────┼────────┼───────┼──────────────────────────────────────┼────────┤
│ ependymoma         │ relative │        3 │ "Ependymoma"[MeSH]            │   3746 │     3 │ {'query.cond': 'Ependymoma'}         │    323 │
│ ganglioglioma      │ relative │        3 │ "Ganglioglioma"[MeSH]         │    865 │     3 │ {'query.cond': 'Ganglioglioma'}      │     15 │
│ gliosarcoma        │ relative │        3 │ "Gliosarcoma"[MeSH]           │    686 │     3 │ {'query.cond': 'Gliosarcoma'}        │    505 │
│ medu

## 2. Drugs: intervention searches

Temozolomide's drug-class neighbours become topics too. Only the drug topics are shown.

In [4]:
drugs = round_of('temozolomide resistance in glioblastoma', max_topics=14, keep=lambda t: 'chemical' in t['types'])

┌────────────────────────────────────────────────────┬─────────────────┬──────────┬───────────────────────────────────────────────────────────────────┬────────┬───────┬───────────────────────────────────────────────────┬────────┐
│                        name                        │     reason      │ distance │                           pubmed query                            │ papers │ take  │                   trial search                    │ trials │
│                      varchar                       │     varchar     │  int64   │                              varchar                              │ int64  │ int64 │                      varchar                      │ int64  │
├────────────────────────────────────────────────────┼─────────────────┼──────────┼───────────────────────────────────────────────────────────────────┼────────┼───────┼───────────────────────────────────────────────────┼────────┤
│ temozolomide                                       │ seed            │        

## 3. Genes, and big neighbours

A gene is searched by MeSH's record for the human protein and its full name, never the bare symbol (PubMed ignores case: `MET[tiab]` would match *met*), and gets no trial search. A topic over `split_cap` is split into its MeSH children only if it's what the goal is about (`split_reasons`); *Adenocarcinoma*, only a see-also neighbour here, is narrowed to `[MeSH:noexp]` instead of splitting into every adenocarcinoma.

In [5]:
genes = round_of('EGFR inhibitors for non-small cell lung cancer', max_topics=8)

┌───────────────────────────────┬──────────┬──────────┬───────────────────────────────────────────────────────────────────────┬────────┬───────┬─────────────────────────────────────────────────┬────────┐
│             name              │  reason  │ distance │                             pubmed query                              │ papers │ take  │                  trial search                   │ trials │
│            varchar            │ varchar  │  int64   │                                varchar                                │ int64  │ int64 │                     varchar                     │ int64  │
├───────────────────────────────┼──────────┼──────────┼───────────────────────────────────────────────────────────────────────┼────────┼───────┼─────────────────────────────────────────────────┼────────┤
│ carcinoma_non_small_cell_lung │ seed     │        0 │ "Carcinoma, Non-Small-Cell Lung"[MeSH]                                │  72754 │     3 │ {'query.cond': 'Non-Small-Cell Lung Car

## From the command line

```
python src/03-topics/run.py --experiment experiments/default.toml           # writes the next round
python src/03-topics/run.py --experiment experiments/default.toml --redo    # rewrites the latest round
```
Each run reads `02-expand/graph_<n>` and writes `runs/<experiment>/03-topics/round_<n>.json`: the topics (every source's search, count and take) and the skipped nodes. Step 04 downloads every round.